In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"
from functools import partial
import time
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy

import matplotlib as mpl
from matplotlib import rc
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10})
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}"

In [ ]:
import jax
import jax.numpy as jnp
# jax.config.update("jax_enable_x64", True)
# jax.config.update("jax_debug_nans", True)
gpus = jax.devices()
print(gpus)

jax.config.update("jax_default_device", gpus[0])

import diffrax
import equinox as eqx
import optax

from haiku import PRNGSequence

from dmpe.data_management import DataPaths
from dmpe.evaluation.plotting_utils import plot_sequence
from dmpe.evaluation.experiment_utils import get_experiment_ids, load_experiment_results
from dmpe.models.models import NeuralEulerODECartpole
from dmpe.models.model_utils import simulate_ahead_with_env

In [ ]:
from dmpe.utils.env_utils.pendulum_utils import setup_env as setup_pendulum_env
from dmpe.utils.env_utils.cart_pole_utils import setup_env as setup_cart_pole_env

In [ ]:
from dmpe.utils.density_estimation import build_grid
from dmpe.models.model_utils import simulate_ahead_with_env

In [ ]:
from dmpe.utils.sets.reachable_set import approximate_reachable_set
from dmpe.utils.sets.control_invariant_set import approximate_control_invariant_set

from dmpe.utils.sets.reachable_set import save_results as save_results_rs
from dmpe.utils.sets.control_invariant_set import save_results as save_results_ci
from dmpe.utils.sets.shared import load_results, DiscretizedSet, SlicedSet

# Reachable set:

In [ ]:
# env, penalty_function, featurize, _ = setup_pendulum_env()
# key = jax.random.PRNGKey(14)
# key, key_rs, _ = jax.random.split(key, 3)

# obs_dim = env.reset(env.env_properties)[0].shape[-1]
# sequence_length = 500
# n_starts = 100
# n_opt_steps = 20_000

# tolerance = 1e-4

# xs = [jnp.linspace(-1.0, 1.0, 50), jnp.linspace(0.8, 1.0, 50)]
# z_g = jnp.meshgrid(*xs, indexing="ij")
# z_g = jnp.stack([_x for _x in z_g], axis=-1)
# target_observations_rs = z_g.reshape(-1, 2)


# chosen_actions_rs, losses_rs, proposed_actions_rs = approximate_reachable_set(
#     env,
#     target_observations_rs,
#     penalty_function,
#     featurize,
#     key=key_rs,
#     sequence_length=sequence_length,
#     n_starts=n_starts,
#     n_opt_steps=n_opt_steps,
# )
# plt.contourf(
#     target_observations_rs.reshape((points_per_dim, points_per_dim, -1))[..., 0],
#     target_observations_rs.reshape((points_per_dim, points_per_dim, -1))[..., 1],
#     jnp.abs(losses_rs.reshape(points_per_dim, points_per_dim)),
# )
# plt.show()

In [ ]:
# save_results_rs(
#     DataPaths().reach_ci_experiments / "pendulum_reachabiltity.json",
#     chosen_actions_rs,
#     losses_rs,
#     target_observations_rs,
# )

In [ ]:
data_rs = load_results(
    DataPaths().reach_ci_experiments / "pendulum_reachabiltity.json",
)

In [ ]:
points_per_dim = int(jnp.sqrt(data_rs["target_observations_rs"].shape[0]))
plt.contourf(
    data_rs["target_observations_rs"].reshape((points_per_dim, points_per_dim, -1))[..., 0],
    data_rs["target_observations_rs"].reshape((points_per_dim, points_per_dim, -1))[..., 1],
    jnp.abs(data_rs["losses_rs"].reshape(points_per_dim, points_per_dim)),
)
plt.show()

In [ ]:
R_s = DiscretizedSet(
    grid=data_rs["target_observations_rs"],
    mask=jnp.abs(data_rs["losses_rs"]) < 1e-5,
    unflattened_shape=tuple([points_per_dim] * 2),
)
R_s

- transfer to actual set representation? see [Baier2012]
- I guess this would be a function that evaluates an input to a bool?

# Control invariant set:

In [ ]:
# env, penalty_function, featurize, _ = setup_pendulum_env()
# key = jax.random.PRNGKey(14)
# key, _, key_ci = jax.random.split(key, 3)

# obs_dim = env.reset(env.env_properties)[0].shape[-1]
# sequence_length = 500
# n_starts = 100
# n_opt_steps = 20_000

# xs = [jnp.linspace(-1.0, 1.0, 50), jnp.linspace(0.8, 1.0, 50)]
# z_g = jnp.meshgrid(*xs, indexing="ij")
# z_g = jnp.stack([_x for _x in z_g], axis=-1)
# init_observations_ci = z_g.reshape(-1, 2)

# chosen_actions_ci, losses_ci, proposed_actions_ci = approximate_control_invariant_set(
#     env,
#     init_observations_ci,
#     penalty_function,
#     key=key_ci,
#     sequence_length=sequence_length,
#     n_starts=n_starts,
#     n_opt_steps=n_opt_steps,
# )

# points_per_dim = int(jnp.sqrt(init_observations_ci.shape[0]))
# plt.contourf(
#      init_observations_ci.reshape((points_per_dim, points_per_dim, -1))[..., 0],
#      init_observations_ci.reshape((points_per_dim, points_per_dim, -1))[..., 1],
#      jnp.abs(losses_ci.reshape(points_per_dim, points_per_dim)),
# )
# plt.show()

In [ ]:
# save_results_ci(
#     DataPaths().reach_ci_experiments / "pendulum_control_invariance.json",
#     chosen_actions_ci,
#     losses_ci,
#     init_observations_ci,
# )

In [ ]:
data_ci = load_results(
    DataPaths().reach_ci_experiments / "pendulum_control_invariance.json",
)

In [ ]:
C = DiscretizedSet(
    grid=data_ci["init_observations_ci"],
    mask=jnp.isclose(jnp.abs(data_ci["losses_ci"]), 0),
    unflattened_shape=tuple([points_per_dim] * 2),
)
C

# Combine:

In [ ]:
data_rs = load_results(
    DataPaths().reach_ci_experiments / "pendulum_reachabiltity.json",
)

data_ci = load_results(
    DataPaths().reach_ci_experiments / "pendulum_control_invariance.json",
)

In [ ]:
points_per_dim = int(jnp.sqrt(data_rs["target_observations_rs"].shape[0]))
plt.contourf(
    data_rs["target_observations_rs"].reshape((points_per_dim, points_per_dim, -1))[..., 0],
    data_rs["target_observations_rs"].reshape((points_per_dim, points_per_dim, -1))[..., 1],
    jnp.abs(data_rs["losses_rs"].reshape(points_per_dim, points_per_dim)),
)
plt.show()

points_per_dim = int(jnp.sqrt(data_ci["init_observations_ci"].shape[0]))
plt.contourf(
    data_ci["init_observations_ci"].reshape((points_per_dim, points_per_dim, -1))[..., 0],
    data_ci["init_observations_ci"].reshape((points_per_dim, points_per_dim, -1))[..., 1],
    jnp.abs(data_ci["losses_ci"].reshape(points_per_dim, points_per_dim)),
)
plt.show()

In [ ]:
S_x = R_s & C

In [ ]:
points_per_dim = int(jnp.sqrt(data_ci["init_observations_ci"].shape[0]))
plt.contourf(
    S_x.grid_unflattened[..., 0],
    S_x.grid_unflattened[..., 1],
    S_x.bool_unflattened,
)
plt.show()

# C(x) to C(x, u)

In [ ]:
observations = data_ci["init_observations_ci"]
C

In [ ]:
R_s.visualize()

In [ ]:
C.visualize()

In [ ]:
S_x.visualize()

In [ ]:
xs = [jnp.linspace(-1.0, 1.0, 50), jnp.linspace(-0.8, 0.8, 50)]
z_g = jnp.meshgrid(*xs, indexing="ij")
z_g = jnp.stack([_x for _x in z_g], axis=-1)
grid = z_g.reshape(-1, 2)

fill = DiscretizedSet(
    grid=grid,
    mask=jnp.ones(shape=grid.shape[:-1], dtype=bool),
    unflattened_shape=(50,50),
)

In [ ]:
inv_S_x = DiscretizedSet(
    grid=-S_x.grid,
    mask=S_x.mask,
    unflattened_shape=S_x.unflattened_shape,
)
inv_S_x.visualize()

In [ ]:
sliced_set = SlicedSet(sets=[S_x, fill, inv_S_x])
for s in sliced_set.sets:
    s.visualize()

In [ ]:
observations = jnp.concatenate([s.grid for s in sliced_set.sets], axis=0)

In [ ]:
actions = jnp.linspace(-1, 1, 50)[..., None]

In [ ]:
@eqx.filter_jit
def step_from_init_obs(init_obs, action, env):
    init_state = env.generate_state_from_observation(init_obs, env.env_properties)
    next_obs, _ = env.step(init_state, action, env.env_properties)
    return next_obs

In [ ]:
env, penalty_function, featurize, _ = setup_pendulum_env()
obs_dim = env.reset(env.env_properties)[0].shape[-1]

# simulate one step ahead
next_observations = eqx.filter_vmap(eqx.filter_vmap(step_from_init_obs, in_axes=(None, 0, None)), in_axes=(0, None, None))(observations, actions, env)

# next_observations.reshape([points_per_dim] * dim + [-1]).shape
next_observations = next_observations.reshape([-1, obs_dim])

In [ ]:
out = eqx.filter_vmap(sliced_set.check_in_set)(next_observations)

In [ ]:
out.shape

In [ ]:
dim = 3
safe = out.reshape([points_per_dim] * dim)
labels=["theta", "omega", "u"]

In [ ]:
fig, axs = plt.subplots(nrows=dim, ncols=dim, figsize=(9, 9), sharex=True, sharey=True)

feature_indices = jnp.arange(0, dim, 1).tolist()
points_per_dim = int(jnp.sqrt(observations.shape[0]))

for i in range(dim):
    for j in range(dim):

        axs[j, i].grid(True)

        reduction_indices = [f_idx for f_idx in feature_indices if not (f_idx == i or f_idx == j)]
        if len(reduction_indices) == dim - 1:
            continue

        any_safe = jnp.any(safe, axis=tuple(reduction_indices))

        if i < j:
            any_safe = jnp.transpose(any_safe)

        axs[j, i].contourf(
            observations.reshape((points_per_dim, points_per_dim, -1))[..., 0],
            observations.reshape((points_per_dim, points_per_dim, -1))[..., 1],
            any_safe,
        )
        
        axs[j, 0].set_ylabel(labels[j])

    axs[-1, i].set_xlabel(labels[i])
fig.tight_layout()

In [ ]:
fig, axs = plt.subplots(nrows=dim, ncols=dim, figsize=(9, 9), sharex=True, sharey=True)

feature_indices = jnp.arange(0, dim, 1).tolist()
points_per_dim = int(jnp.sqrt(observations.shape[0]))

for i in range(dim):
    for j in range(dim):

        axs[j, i].grid(True)

        reduction_indices = [f_idx for f_idx in feature_indices if not (f_idx == i or f_idx == j)]
        if len(reduction_indices) == dim - 1:
            continue

        all_safe = jnp.all(safe, axis=tuple(reduction_indices))

        if i < j:
            all_safe = jnp.transpose(all_safe)

        axs[j, i].contourf(
            observations.reshape((points_per_dim, points_per_dim, -1))[..., 0],
            observations.reshape((points_per_dim, points_per_dim, -1))[..., 1],
            all_safe,
        )
        
        axs[j, 0].set_ylabel(labels[j])

    axs[-1, i].set_xlabel(labels[i])
fig.tight_layout()

In [ ]:
out_ = np.array(out.reshape([points_per_dim] * dim))

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

ax.voxels(
    out_,
    alpha=0.8,
)
ax.view_init(azim=60, elev=30)
plt.show()

In [ ]:
test = jnp.any(safe, axis=-1)

In [ ]:
points_per_dim = int(jnp.sqrt(data_ci["init_observations_ci"].shape[0]))
plt.contourf(
    data_ci["init_observations_ci"].reshape((points_per_dim, points_per_dim, -1))[..., 0],
    data_ci["init_observations_ci"].reshape((points_per_dim, points_per_dim, -1))[..., 1],
    test,
)
plt.show()

points_per_dim = int(jnp.sqrt(data_ci["init_observations_ci"].shape[0]))
plt.contourf(
    data_ci["init_observations_ci"].reshape((points_per_dim, points_per_dim, -1))[..., 0],
    data_ci["init_observations_ci"].reshape((points_per_dim, points_per_dim, -1))[..., 1],
    C,
)
plt.show()

points_per_dim = int(jnp.sqrt(data_ci["init_observations_ci"].shape[0]))
plt.contourf(
    data_ci["init_observations_ci"].reshape((points_per_dim, points_per_dim, -1))[..., 0],
    data_ci["init_observations_ci"].reshape((points_per_dim, points_per_dim, -1))[..., 1],
    jnp.logical_xor(C, test),
)
plt.show()

# Combine to $R_s^{(x)}$ and $C^{(x, u)}$ to $S^{(x, u)}$

In [ ]:
C_xu = safe

In [ ]:
R_s_xu = R_s[..., None].repeat(C_xu.shape[-1], axis=-1)

In [ ]:
S_xu = jnp.logical_and(C_xu, R_s_xu)

In [ ]:
S_xu.shape

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

ax.voxels(
    np.array(S_xu),
    alpha=1.0,  
)
ax.view_init(azim=60, elev=30)
plt.show()

In [ ]:
fig, axs = plt.subplots(nrows=dim, ncols=dim, figsize=(9, 9), sharex=True, sharey=True)

feature_indices = jnp.arange(0, dim, 1).tolist()
points_per_dim = int(jnp.sqrt(observations.shape[0]))

for i in range(dim):
    for j in range(dim):

        axs[j, i].grid(True)

        reduction_indices = [f_idx for f_idx in feature_indices if not (f_idx == i or f_idx == j)]
        if len(reduction_indices) == dim - 1:
            continue

        any_safe = jnp.any(S_xu, axis=tuple(reduction_indices))

        if i < j:
            any_safe = jnp.transpose(any_safe)

        axs[j, i].contourf(
            observations.reshape((points_per_dim, points_per_dim, -1))[..., 0],
            observations.reshape((points_per_dim, points_per_dim, -1))[..., 1],
            any_safe,
        )
        
        axs[j, 0].set_ylabel(labels[j])

    axs[-1, i].set_xlabel(labels[i])
fig.tight_layout()


fig, axs = plt.subplots(nrows=dim, ncols=dim, figsize=(9, 9), sharex=True, sharey=True)

feature_indices = jnp.arange(0, dim, 1).tolist()
points_per_dim = int(jnp.sqrt(observations.shape[0]))

for i in range(dim):
    for j in range(dim):

        axs[j, i].grid(True)

        reduction_indices = [f_idx for f_idx in feature_indices if not (f_idx == i or f_idx == j)]
        if len(reduction_indices) == dim - 1:
            continue

        all_safe = jnp.all(S_xu, axis=tuple(reduction_indices))

        if i < j:
            all_safe = jnp.transpose(all_safe)

        axs[j, i].contourf(
            observations.reshape((points_per_dim, points_per_dim, -1))[..., 0],
            observations.reshape((points_per_dim, points_per_dim, -1))[..., 1],
            all_safe,
        )
        
        axs[j, 0].set_ylabel(labels[j])

    axs[-1, i].set_xlabel(labels[i])
fig.tight_layout()
plt.show()